# P0 · Python refresher (pre-work)

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Self-study before Day 1 | about 45 minutes | [Course overview](../README.md) | [00 · Warm-up](00_warmup_whats_wrong.ipynb) |

This notebook is optional. It refreshes the plain Python and pandas you will use throughout the workshop, so that on Day 1 you can focus on the new ideas (classes, objects, design) instead of syntax. If everything here feels easy, skip straight to the self-assessment at the end.

> **Learning goals.** By the end you can:
>
> - write functions with default and keyword arguments and explain what they return
> - use lists, dictionaries and comprehensions to reshape small pieces of data
> - load the workshop dataset with pandas and answer simple questions about it
> - count, filter and average with pandas and NumPy the way the workshop code does
> - read a traceback and find the line that caused an error

**Before you start.** Run `uv sync` in the repository root and select the `.venv` kernel (see the README).

**In this notebook**

1. Theory 1 · Functions
2. Hands-on A · Quick churn estimates
3. Theory 2 · Lists, dictionaries and comprehensions
4. Hands-on B · Customer lists and lookups
5. Theory 3 · f-strings
6. Theory 4 · pandas essentials
7. Theory 5 · NumPy arrays and model outputs
8. Hands-on C · Exploring the churn data
9. Theory 6 · `assert`, tracebacks and errors
10. Hands-on D · A column stored as text

---

## Theory 1 · Functions

A function packages a piece of logic under a name. Parameters can have **default values**, and callers can pass arguments **by keyword**, which makes calls easier to read.

> **Reading note: type hints and docstrings.** In the code below, `n_churned: int` says the parameter should be an integer and `-> float` says the function returns a float. Python does not check these hints when it runs the code; they are there for readers and for tools such as your editor. The text in triple quotes right under `def` is a **docstring**: a short description that `help(churn_rate)` shows. Notebook 07 covers both properly.

In [ ]:
def churn_rate(n_churned: int, n_customers: int) -> float:
    """Share of customers who cancelled their contract."""
    return n_churned / n_customers


def monthly_revenue(n_customers: int, avg_charge: float = 64.76, discount: float = 0.0) -> float:
    """Revenue for one month, optionally after a discount (0.1 means 10 %)."""
    return n_customers * avg_charge * (1 - discount)


print(churn_rate(1869, 7043))
print(monthly_revenue(1000))  # uses both defaults
print(monthly_revenue(1000, discount=0.1))  # keyword argument, avg_charge keeps its default

A function without a `return` statement returns `None`. That is a common source of confusion: printing inside a function is not the same as returning a value.

In [ ]:
def show_rate(n_churned: int, n_customers: int) -> None:
    print(f"churn rate: {n_churned / n_customers:.1%}")


result = show_rate(1869, 7043)
print(result)  # None: the function printed something but returned nothing

---

## Hands-on A · Quick churn estimates

The retention team often asks for back-of-the-envelope numbers: "if a quarter of these customers leave, how much revenue do we lose?" A small function answers that question the same way every time.

### Exercise A.1

*About 5 minutes, including the discussion.*

Write `expected_lost_revenue(n_customers, churn_rate, avg_charge=64.76, months=12)` that returns how much monthly revenue is lost over `months` months if `churn_rate` of `n_customers` customers leave.

**What your code should do.**

- `n_customers` is a whole number of customers, `churn_rate` a share between 0 and 1 (0.25 means 25 %), `avg_charge` the average monthly bill, `months` how many months we look ahead.
- Return (do not print) one number: lost customers × monthly charge × months.
- Example: `expected_lost_revenue(100, 0.25)` returns `19428.0` (25 customers × 64.76 × 12).

The starter cell contains `raise NotImplementedError("Your code here")`. It deliberately stops with an error until you replace it with your code (Theory 6 explains errors).

**Discuss.** Which parameters deserve a default value, and which should the caller always pass? What could go wrong if `churn_rate` had a default too?

In [ ]:
# Exercise A.1: your code here


def expected_lost_revenue(n_customers, churn_rate, avg_charge=64.76, months=12):
    raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise A.1: run after solving it
assert round(expected_lost_revenue(100, 0.25), 2) == 19428.0
assert expected_lost_revenue(100, 0.25, avg_charge=10, months=1) == 250
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
def expected_lost_revenue(n_customers, churn_rate, avg_charge=64.76, months=12):
    return n_customers * churn_rate * avg_charge * months
```
</details>

---

## Theory 2 · Lists, dictionaries and comprehensions

A **list** holds items in order. A **dictionary** maps keys to values. A **comprehension** builds a new list or dictionary from an existing one in a single line.

In [ ]:
contracts = ["Month-to-month", "One year", "Two year"]
churn_by_contract = {"Month-to-month": 0.427, "One year": 0.113, "Two year": 0.028}

for contract in contracts:
    print(f"{contract:<15} {churn_by_contract[contract]:.1%}")

# list comprehension: keep only the risky contracts
risky = [c for c in contracts if churn_by_contract[c] > 0.1]
print(risky)

# dict comprehension: churn rate in percent, rounded
in_percent = {c: round(rate * 100, 1) for c, rate in churn_by_contract.items()}
print(in_percent)

---

## Hands-on B · Customer lists and lookups

Data often arrives as a plain list of records before it ever becomes a DataFrame. Here you turn a small list of `(customer_id, monthly_charge)` pairs into the two structures you need most: a filtered list and a lookup table.

### Exercise B.1

*About 5 minutes, including the discussion.*

Given the list `charges` (in the next cell), build:

1. `expensive`: the ids of customers paying more than 80 per month, in the original order.
2. `charge_by_id`: a dictionary from id to monthly charge.

**What your code should do.**

- `expensive` is a list of id strings, only for charges above 80, in the same order as in `charges`. With the data below: `["6713-OKOMC"]`.
- `charge_by_id` is a dictionary with 4 entries, e.g. `charge_by_id["5575-GNVDE"]` gives `56.95`.
- Each pair unpacks into two names in a loop or comprehension: `for cid, charge in charges: ...`

**Discuss.** A list of pairs or a dictionary: when is each the better structure? What happens to `charge_by_id` if the same customer id appears twice in `charges`?

In [ ]:
charges = [
    ("7590-VHVEG", 29.85),
    ("5575-GNVDE", 56.95),
    ("9237-HQITU", 70.70),
    ("6713-OKOMC", 99.65),
]

In [ ]:
# Exercise B.1: your code here

expensive = None  # your code here
charge_by_id = None  # your code here

In [ ]:
# Check for exercise B.1: run after solving it
assert expensive == ["6713-OKOMC"]
assert charge_by_id["5575-GNVDE"] == 56.95
assert len(charge_by_id) == 4
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
expensive = [cid for cid, charge in charges if charge > 80]
charge_by_id = {cid: charge for cid, charge in charges}
# or simply: charge_by_id = dict(charges)
```
</details>

---

## Theory 3 · f-strings

f-strings put values into text. After a colon you can format numbers: `:.2f` (two decimals), `:.1%` (percentage), `:,` (thousands separator), `:>8` (right-align in 8 characters).

In [ ]:
n, churned, revenue = 7043, 1869, 456116.6
print(f"{churned} of {n} customers churned ({churned / n:.1%}).")
print(f"Monthly revenue: {revenue:,.2f}")
print(f"|{'tenure':>8}|{'charges':>8}|")

---

## Theory 4 · pandas essentials

pandas stores a table as a **DataFrame**: rows plus named columns. One column on its own is a **Series**. The workshop uses a handful of pandas operations again and again; here they are on a table of six customers, small enough to check every result by eye. Come back to this section whenever a pandas line in a later notebook puzzles you.

In [ ]:
import pandas as pd

small = pd.DataFrame(
    {
        "customer": ["A", "B", "C", "D", "E", "F"],
        "Contract": [
            "Month-to-month",
            "Two year",
            "Month-to-month",
            "One year",
            "Month-to-month",
            "Two year",
        ],
        "tenure": [2, 60, 5, 30, 14, 45],
        "MonthlyCharges": [70.5, 25.0, 89.9, 55.3, 80.0, 20.0],
        "churned": [1, 0, 1, 0, 0, 0],  # 1 = the customer cancelled, 0 = stayed
    }
)
small

**Selecting columns.** One name in brackets gives a Series, a list of names gives a smaller DataFrame.

In [ ]:
print(small["tenure"])  # a Series: one column
small[["customer", "tenure"]]  # a DataFrame: the listed columns

**Boolean masks.** A comparison on a column gives one `True`/`False` per row. Used inside the brackets, it keeps the rows where the value is `True`. Because `True` counts as 1, `.sum()` counts the rows that match and `.mean()` gives their share. Combine conditions with `&` (and) or `|` (or), each condition in its own parentheses.

In [ ]:
mask = small["tenure"] < 12  # True for customers with less than a year
print(mask.tolist())
print(small[mask]["customer"].tolist())  # keep only the rows where mask is True
print(mask.sum())  # how many rows match: 2

new_and_expensive = (small["tenure"] < 12) & (small["MonthlyCharges"] > 80)
print(small[new_and_expensive]["customer"].tolist())

**The mean of a 0/1 column is a rate.** With `churned` coded as 1 and 0, `.mean()` is the share of customers who churned. Text answers such as `"Yes"`/`"No"` become 0/1 with a comparison and `.astype(int)`.

In [ ]:
print(small["churned"].mean())  # 2 of 6 churned: 0.333...

answers = pd.Series(["Yes", "No", "Yes"])
print((answers == "Yes").astype(int).tolist())  # [1, 0, 1]

**Counting and grouping.** `value_counts()` counts how often each value appears (`normalize=True` gives shares instead). `groupby("Contract")["churned"].mean()` splits the rows by contract type and computes one churn rate per group; the result is a Series indexed by contract type.

In [ ]:
print(small["Contract"].value_counts())
print(small["Contract"].value_counts(normalize=True))

rate_by_contract = small.groupby("Contract")["churned"].mean()
print(rate_by_contract)
print(rate_by_contract["Month-to-month"])  # look up one group: 2 of 3

**Rows and a column at once with `.loc`.** `df.loc[rows, column]` selects rows (here with a mask) and one column in a single step.

In [ ]:
small.loc[small["churned"] == 1, "MonthlyCharges"].mean()  # average bill of the churners

> **Check your understanding**
>
> ```python
> print((small["tenure"] >= 30).mean())
> ```
>
> What is printed?
>
> <details><summary><b>Show answer</b></summary>
>
> `0.5`. The comparison gives `True` for B (60), D (30) and F (45), three of six rows. The mean of `True`/`False` values is the share of `True`s.
> </details>

In [ ]:
# Predict the output first, then run this cell
print((small["tenure"] >= 30).mean())

---

## Theory 5 · NumPy arrays and model outputs

scikit-learn returns its predictions as **NumPy arrays**, not as pandas objects. An array behaves much like a column: comparisons work element by element, `&` combines conditions, and `.sum()` and `.mean()` count and average. `series.to_numpy()` turns a pandas column into an array.

In [ ]:
import numpy as np

y_true = np.array([1, 0, 1, 1, 0, 0])  # who really churned
y_pred = np.array([1, 0, 0, 1, 1, 0])  # who a model flagged

print(y_true == y_pred)  # element by element: True where the model was right
print((y_true == y_pred).mean())  # share of correct predictions: 4 of 6

caught = (y_true == 1) & (y_pred == 1)  # churners the model flagged
print(caught.sum())  # how many: 2

**What `predict_proba` returns.** A classifier's `predict_proba(X)` gives a 2-dimensional array: one row per customer, one column per class. Column 0 is the probability of "stays" (class 0), column 1 the probability of "churns" (class 1), and each row adds up to 1. `proba[:, 1]` means "all rows, column 1": the churn probability of every customer. Compare it with a threshold to decide whom to flag.

In [ ]:
proba = np.array(
    [[0.9, 0.1], [0.2, 0.8], [0.6, 0.4], [0.35, 0.65]]
)  # what predict_proba returns for 4 customers

churn_risk = proba[:, 1]  # all rows, column 1
print(churn_risk)
flagged = (churn_risk >= 0.5).astype(int)  # 1 where the risk is at least 50 %
print(flagged)
print(np.where(churn_risk >= 0.5, "call", "skip"))  # pick one of two values per element

> **Check your understanding**
>
> ```python
> print(((y_true == 1) & (y_pred == 0)).sum())
> ```
>
> What is printed, and what does the number mean?
>
> <details><summary><b>Show answer</b></summary>
>
> `1`. Only the third customer churned (`y_true` is 1) without being flagged (`y_pred` is 0). In the language of notebook 01, that is a **false negative**: a churner the model missed.
> </details>

In [ ]:
# Predict the output first, then run this cell
print(((y_true == 1) & (y_pred == 0)).sum())

---

## Hands-on C · Exploring the churn data

Time for the real dataset: the **Telco customer churn** table, one row per customer of a telecom company, with their contract, services, charges and whether they cancelled (`Churn`). The next cell loads it and adds a 0/1 column `churned`, exactly as in Theory 4.

In [ ]:
from pathlib import Path

DATA_PATH = Path("../data/raw/telco_churn.csv")
customers = pd.read_csv(DATA_PATH)
customers["churned"] = (customers["Churn"] == "Yes").astype(int)

print(customers.shape)  # (rows, columns)
customers.head(3)

### Exercise C.1

*About 5 minutes, including the discussion.*

1. Compute `avg_charge_churned`: the mean `MonthlyCharges` of customers who churned.
2. Compute `churn_by_internet`: the churn rate per `InternetService` value, as a pandas Series.

**What your code should do.**

- `avg_charge_churned` is a single number, about 74.44.
- `churn_by_internet` is a Series with one entry per internet service (`DSL`, `Fiber optic`, `No`); each value is a churn rate between 0 and 1, e.g. about 0.419 for `Fiber optic`.

**pandas and NumPy you need.**

- `customers.loc[customers["churned"] == 1, "MonthlyCharges"]`: the monthly charges of the churners only; add `.mean()` for their average.
- `customers.groupby("InternetService")["churned"].mean()`: one churn rate per internet service.

**Discuss.** Churners pay more per month on average. Does that mean high prices cause churn? What else could explain it? (Look at `churn_by_internet`.)

In [ ]:
# Exercise C.1: your code here

avg_charge_churned = None  # your code here
churn_by_internet = None  # your code here

In [ ]:
# Check for exercise C.1: run after solving it
assert round(avg_charge_churned, 2) == 74.44
assert round(churn_by_internet["Fiber optic"], 3) == 0.419
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
avg_charge_churned = customers.loc[customers["Churn"] == "Yes", "MonthlyCharges"].mean()
churn_by_internet = customers.groupby("InternetService")["churned"].mean()
```
</details>

### Exercise C.2

*About 10 minutes, including the discussion.*

The retention team's current rule of thumb is "call every customer on a month-to-month contract". How good is that rule? The next cell turns the truth and the rule into two NumPy arrays of 0s and 1s. Count:

- `tp`: churners the rule flags (true positives),
- `fp`: customers the rule flags who stayed (false positives),
- `fn`: churners the rule misses (false negatives),
- `tn`: customers the rule leaves alone who stayed (true negatives),

and compute `recall = tp / (tp + fn)`, the share of churners the rule reaches.

**What your code should do.**

- `tp`, `fp`, `fn` and `tn` are whole numbers that add up to 7043 (every customer lands in exactly one group).
- `recall` is a number between 0 and 1.

**pandas and NumPy you need.**

- `(y_true == 1) & (y_pred == 1)`: `True` where both conditions hold; `.sum()` counts the `True`s.
- Change the `1`s and `0`s for the other three counts.

**Discuss.** The rule reaches most churners. How many customers would the team have to call, and how many of those calls go to people who would have stayed anyway? Would you keep the rule? Notebook 01 bundles these four numbers into one object.

In [ ]:
# Given: the truth and the rule as arrays of 0s and 1s
y_true = customers["churned"].to_numpy()
y_pred = (
    (customers["Contract"] == "Month-to-month").astype(int).to_numpy()
)  # 1 = the rule says "call"

In [ ]:
# Exercise C.2: your code here

tp = None
fp = None
fn = None
tn = None
recall = None

In [ ]:
# Check for exercise C.2: run after solving it
assert tp + fp + fn + tn == 7043
assert (tp, fn) == (1655, 214)
assert round(recall, 3) == 0.886
print(f"The rule calls {tp + fp} customers and reaches {recall:.1%} of the churners")
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
tp = ((y_true == 1) & (y_pred == 1)).sum()
fp = ((y_true == 0) & (y_pred == 1)).sum()
fn = ((y_true == 1) & (y_pred == 0)).sum()
tn = ((y_true == 0) & (y_pred == 0)).sum()
recall = tp / (tp + fn)
```

The rule reaches 88.6 % of the churners, but it calls 3875 customers, and 2220 of those calls (57 %) go to customers who would have stayed. High recall alone does not make a rule good: every call costs money.
</details>

---

## Theory 6 · `assert`, tracebacks and errors

`assert condition, "message"` stops the program with an `AssertionError` if the condition is false. The check cells in this workshop use it, and it is the basis of testing with pytest on Day 2.

When something goes wrong, Python prints a **traceback**. Read it **from the bottom up**: the last line names the error type and message, the lines above show where it happened, with the most recent call last.

> **Reading note: errors on purpose.** `raise SomeError("message")` stops the code with that error and message. The starter cells of the exercises use `raise NotImplementedError("Your code here")` for this: they fail until you write your code. `try: ... except SomeError as exc: ...` runs the `except` block instead of stopping when that error happens; `exc` holds the error, so you can print it. The cells below use it to show an error without stopping the notebook. The built-in errors you will meet most: `ValueError` (right type, wrong value), `TypeError` (wrong type), `KeyError` (missing key or column), `RuntimeError` (something was used in the wrong state). Notebook 09 covers errors properly.

In [ ]:
rate = 1869 / 7043
assert 0 <= rate <= 1, "a churn rate must be between 0 and 1"  # true, so nothing happens

charges_as_text = pd.Series(["29.85", "56.95", " "])  # numbers stored as text, one of them blank

try:
    charges_as_text.mean()
except TypeError as exc:
    print(f"{type(exc).__name__}: {exc}")

pandas cannot average text, even when the text looks like numbers. `pd.to_numeric` converts it; with `errors="coerce"`, anything that is not a number (here the blank) becomes `NaN` ("not a number", pandas' marker for a missing value) instead of raising an error. `.isna()` then gives `True` for every missing value.

In [ ]:
numbers = pd.to_numeric(charges_as_text, errors="coerce")
print(numbers.tolist())  # [29.85, 56.95, nan]
print(numbers.isna().sum())  # how many values are missing: 1
print(numbers.mean())  # pandas skips NaN when averaging

---

## Hands-on D · A column stored as text

The real dataset has the same problem. Averaging `TotalCharges` fails:

In [ ]:
def average_charge(df: pd.DataFrame) -> float:
    return df["TotalCharges"].mean()


try:
    average_charge(customers)
except TypeError as exc:
    print(f"{type(exc).__name__}: {exc}")

`TotalCharges` is stored as **text** in this file (because a few values are blank), so pandas cannot average it. You will meet this exact problem again in the warm-up notebook.

### Exercise D.1

*About 5 minutes, including the discussion.*

Convert `TotalCharges` to numbers, store the result in `total`, and count how many values are missing in `n_missing`.

**What your code should do.**

- `total` is a Series of floats with one value per customer (7043 values), `NaN` where `TotalCharges` was blank.
- `n_missing` is a whole number: how many values of `total` are `NaN`.

**pandas and NumPy you need.**

- `pd.to_numeric(customers["TotalCharges"], errors="coerce")`: text to numbers, blanks become `NaN`.
- `.isna().sum()`: count the missing values of a Series.

**Discuss.** `errors="coerce"` turned 11 values into `NaN` without complaint. Is that a good default? What would you want to know about those 11 customers before deciding what to do with them?

In [ ]:
# Exercise D.1: your code here

total = None  # your code here
n_missing = None  # your code here

In [ ]:
# Check for exercise D.1: run after solving it
assert total.dtype == float
assert n_missing == 11
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
total = pd.to_numeric(customers["TotalCharges"], errors="coerce")
n_missing = total.isna().sum()
```
</details>

---

## Self-assessment

You are ready for Day 1 if you can, without looking things up:

- [ ] write a function with a default argument and call it with a keyword argument
- [ ] explain the difference between `print` and `return`
- [ ] build a list or dict with a comprehension
- [ ] load a CSV with pandas, filter rows with a boolean mask, and compute a mean per group
- [ ] count matching elements of two NumPy arrays, and take the churn column of a `predict_proba` result
- [ ] find the failing line in a traceback

If two or more boxes stay empty, spend another 30 minutes with the resources below. You will still be able to follow the workshop: a partner or the trainer will help, and the first exercise of every hands-on block is designed for you. Theory 4 and 5 of this notebook stay useful as a pandas and NumPy reference during the workshop.

---

## Sources and further reading

**Sources.** This notebook is based on:

- [The Python Tutorial](https://docs.python.org/3/tutorial/), chapters 3 to 5 (numbers, strings, lists, control flow, functions, data structures).
- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html), the official quick tour.

**For self-study.** Where to go next if you want to learn more:

- [Real Python: Understanding the Python Traceback](https://realpython.com/python-traceback/) (article, beginner): how to read an error message from the bottom up.
- [Wes McKinney, *Python for Data Analysis*, 3rd edition](https://wesmckinney.com/book/) (free online book, beginner to intermediate): pandas from its author. Chapters 2, 3, 5 and 10 cover everything the workshop assumes.

---

[Course overview](../README.md) &nbsp;|&nbsp; [00 · Warm-up →](00_warmup_whats_wrong.ipynb)